<a href="https://colab.research.google.com/github/jaloaiza/genaiassignments/blob/main/assignments/04/Module4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install & Setup Replicate Token

In [1]:
!uv pip install replicate

Using Python 3.12.12 environment at: /usr
Resolved 13 packages in 599ms
Prepared 1 package in 35ms
Installed 1 package in 3ms
 + replicate==1.0.7


In [2]:
import sys
import os
from replicate.client import Client
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['REPLICATE_API_TOKEN'] = userdata.get('REPLICATE_API_TOKEN')
  print("Replicate API Token set for Colab")
else:
  load_dotenv()
  print("Loaded env vars from .env")

Replicate API Token set for Colab


# Prepping for Replicate

## Model

In [3]:
MODEL = "lucataco/qwen3-vl-8b-instruct:39e893666996acf464cff75688ad49ac95ef54e9f1c688fbc677330acc478e11"

## System Prompt

In [4]:
SYSTEM_PROMPT = (
    "You are an accessibility assistant that writes helpful alt-text for blind or low-vision users.\n"
    "Rules:\n"
    "- Be factual and neutral. Do not guess hidden details.\n"
    "- Mention: main subjects, actions, setting, notable text, and spatial layout if helpful.\n"
    "- If the image is unclear, say what is uncertain.\n"
    "- Keep it concise unless asked for detail.\n"
)

## Gradio -> Function -> Replicate

In [5]:
import replicate

def vlm_caption(image, user_prompt, detail_level):
  """
  image: PIL Image from Gradio
  user_prompt: text prompt for the user
  detail_level: Gradio choice
  """

  # Converting the PIL into a bytes buffer
  import io
  buf = io.BytesIO()
  image.save(buf, format="PNG")
  buf.seek(0)

  # Padding the prompt up
  detail_instruction = {
     "Short": "Write 1-2 sentences of alt-text",
     "Medium": "Write a short paragraph of alt text (3-5 sentences)",
     "Detailed": "Write a detailed description with key objects, actions,"
     " scene context and any readable text"
  }

  prompt = f"""
    You are describing an image for accessibility.

    Detail level: {detail_level}
    Guidance: {detail_instruction[detail_level]}

    Focus on what is visually present. Use natural language.
    Avoid lists or headings.

    User request: {user_prompt or "Describe the image."}
  """

  out = replicate.run(
        MODEL,
        input={
            "image": buf,
            "prompt": prompt,
        }
    )

  # Some models return a list of strings (stream); join safely
  if isinstance(out, (list, tuple)):
        return "".join(out)
  return str(out)

In [6]:
import os
import gradio as gr

with gr.Blocks(title="Accessibility Captioning (VLM)") as demo:
    gr.Markdown("# Accessibility Captioning (Replicate VLM)\nUpload an image and generate alt-text.")

    with gr.Row():
        img = gr.Image(type="pil", label="Image")
        with gr.Column():
            user_prompt = gr.Textbox(
                label="Optional prompt",
                placeholder="e.g., 'Describe this for a visually impaired user. Include any readable text.'",
                lines=3
            )
            detail = gr.Radio(["Short", "Medium", "Detailed"], value="Medium", label="Detail level")
            btn = gr.Button("Generate description")

    output = gr.Textbox(label="Generated description", lines=10)

    btn.click(fn=vlm_caption, inputs=[img, user_prompt, detail], outputs=output)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://05d905e31ff8410ce6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://05d905e31ff8410ce6.gradio.live


# Analysis

I chose Option 3: Accessibility. The reason being is I am interested to see the capability and reliability of VLMs to accurately describe an image to a "visually impaired" user.

In my runs of the model, I have unfortunately had almost zero success to this. I have messed around with a little bit of the prompt engineering, and have additionally hit a brick wall. I cannot get the model to accurately describe an image. It talks A LOT about scenery that isn't present due to it mistaking it. And doesn't really describe the foreground of the image.

Additionally, I have had a lot of performance issues of the image analysis. I have tried 3 models and have not had luck of it running under 60 seconds. At first I thought it was a model issue, now I am starting to think it might be going through Replicate.